In [72]:
# Assignment 1: Real-Time Weather Alerts Agent ADK

In [85]:
# Dependencies
from google.colab import userdata
import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

In [ ]:
# API Keys
os.environ["GOOGLE_API_KEY"] = "removed_for_security"
os.environ["GOOGLE_MAPS_API_KEY"] = "removed_for_security"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

MAPS_API_KEY = os.environ["GOOGLE_MAPS_API_KEY"]

print("✅ Setup complete")

✅ Setup complete


In [87]:
# Model to use
MODEL = "gemini-2.5-flash-lite" #switch between "gemini-2.5-flash-lite" or some other lighter model if you get rate limit error

In [88]:
# Tool: Get Extended Weather Forcast
def get_extended_weather_forecast(latitude: float, longitude: float) -> dict:
    """
    Retrieves the extended weather forecast and active alerts for a location
    using the National Weather Service (NWS) API. No API key required.

    Args:
        latitude: The latitude of the location as a float.
        longitude: The longitude of the location as a float.

    Returns:
        A dictionary containing the weather forecast periods and any active
        weather alerts for the given coordinates.
    """
    try:
        # Step 1: Get the NWS grid point for these coordinates
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        headers = {"User-Agent": "WeatherAlertsAgent/1.0 (your@email.com)"}

        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        points_data = points_response.json()

        # Step 2: Get the forecast using the grid info
        forecast_url = points_data["properties"]["forecast"]
        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()

        # Extract the first 3 forecast periods (today, tonight, tomorrow)
        periods = forecast_data["properties"]["periods"][:3]
        forecast_summary = [
            {
                "period": p["name"],
                "temperature": f"{p['temperature']}°{p['temperatureUnit']}",
                "forecast": p["shortForecast"],
                "detail": p["detailedForecast"],
            }
            for p in periods
        ]

        # Step 3: Get active alerts for this area
        zone = points_data["properties"]["forecastZone"].split("/")[-1]
        alerts_url = f"https://api.weather.gov/alerts/active/zone/{zone}"
        alerts_response = requests.get(alerts_url, headers=headers, timeout=10)
        alerts_data = alerts_response.json()

        active_alerts = [
            {
                "event": a["properties"]["event"],
                "severity": a["properties"]["severity"],
                "headline": a["properties"]["headline"],
                "description": a["properties"]["description"][:300],  # trim long text
            }
            for a in alerts_data.get("features", [])
        ]

        return {
            "status": "success",
            "forecast": forecast_summary,
            "alerts": active_alerts if active_alerts else ["No active alerts"],
        }

    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": str(e)}
    except KeyError as e:
        return {"status": "error", "message": f"Unexpected API response format: {e}"}

In [89]:
# Tool: Get Longitude And Lattitude
def get_lat_lon(city: str, state: str) -> dict:
    """
    Converts a US city and state name into geographic coordinates (latitude
    and longitude) using the Google Maps Geocoding API.

    Args:
        city: The name of the city as a string (e.g., "Austin").
        state: The two-letter US state abbreviation (e.g., "TX").

    Returns:
        A dictionary containing the latitude and longitude of the city,
        or an error message if the location could not be found.
    """
    try:
        api_key = os.environ.get("GOOGLE_MAPS_API_KEY")
        query = f"{city}, {state}, USA"
        url = "https://maps.googleapis.com/maps/api/geocode/json"

        params = {"address": query, "key": api_key}
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data["status"] != "OK":
            return {
                "status": "error",
                "message": f"Geocoding failed: {data['status']} for '{query}'"
            }

        location = data["results"][0]["geometry"]["location"]
        return {
            "status": "success",
            "city": city,
            "state": state,
            "latitude": location["lat"],
            "longitude": location["lng"],
        }

    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": str(e)}

In [90]:
#----------------------------------------AGENT DEFINITIONS-------------------------------------------------------
# ── Agent Instructions ────────────────────────────────────────────────────
WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly and helpful weather assistant for the United States.

When a user asks about weather for a city:
1. FIRST use get_lat_lon to convert the city name to coordinates.
2. THEN use get_extended_weather_forecast with those coordinates to get the weather.
3. Present the results in a clear, friendly format including:
   - Any active weather alerts (emphasize these if present)
   - The forecast for today, tonight, and tomorrow
   - A brief recommendation (e.g., "bring an umbrella")

Always be conversational and helpful. If there are active alerts, make sure
to highlight them prominently as they may indicate safety concerns.
If a location is outside the US, politely explain that you only support
US locations (NWS only covers the US).
"""

# ── Option A: Gemini Agent (default) ──────────────────────────────────────
weather_agent_gemini = Agent(
    name="Pat",
    model= MODEL,
    description="Pat the Friendly Weather Agent — powered by Gemini.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],  # plain functions — ADK wraps them
)

In [91]:
# ── Helper function to run a query ────────────────────────────────────────
async def ask_weather(agent, question: str, session_id: str = "test_session") -> str:
    """Run a single question through the weather agent and return the response."""
    session_service = InMemorySessionService()

    runner = Runner(
        agent=agent,
        app_name="weather_app",
        session_service=session_service,
    )

    session = await session_service.create_session(
        app_name="weather_app",
        user_id="test_user",
        session_id=session_id,
    )

    response_text = ""
    async for event in runner.run_async(
        user_id="test_user",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=question)]
        ),
    ):
        if event.is_final_response():
            # ✅ Guard against None content
            if event.content and event.content.parts and event.content.parts[0].text:
                response_text = event.content.parts[0].text
            else:
                response_text = "(No response received — likely a rate limit or empty reply)"

    return response_text

In [92]:
# Helper function to run tests
test_cities = [
    "What's the weather in Austin, TX?",
    "Are there any weather alerts for Miami, FL?",
    "Give me the forecast for Chicago, IL",
]

async def run_all_tests():
    print("=" * 60)
    print("TESTING WEATHER AGENT — GEMINI MODEL")
    print("=" * 60)

    for i, question in enumerate(test_cities):
        print(f"\n🌤️ Query {i+1}: {question}")
        print("-" * 40)
        response = await ask_weather(
            agent=weather_agent_gemini,
            question=question,
            session_id=f"session_{i}"
        )
        print(response)
        print()

        if i < len(test_cities) - 1:  # Don't sleep after last query
            print("⏳ Waiting 15s to avoid rate limits...")
            await asyncio.sleep(15)

In [93]:
# Run tests
await run_all_tests()

TESTING WEATHER AGENT — GEMINI MODEL

🌤️ Query 1: What's the weather in Austin, TX?
----------------------------------------
No active alerts in Austin, TX.

Here's the weather forecast:

**Today:** Expect it to be partly sunny with a high near 82°F. The south-southeast wind will be light.

**Tonight:** It will be mostly cloudy with a low around 63°F. The south-southeast wind will be around 5 mph.

**Tuesday:** Mostly sunny with a high near 88°F. The south wind will be between 5 to 10 mph, with gusts up to 25 mph.

It looks like a warm few days ahead! You might want to stay hydrated.

⏳ Waiting 15s to avoid rate limits...

🌤️ Query 2: Are there any weather alerts for Miami, FL?
----------------------------------------
I found an active weather alert for Miami, FL:

**DANGEROUS RIP CURRENTS ARE EXPECTED** from 10 AM this morning through Wednesday evening. This could sweep even the best swimmers away from the shore into deeper water.

Here's the forecast:

*   **Today:** There's a chance